In [29]:
from co3d.dataset.data_types import (
    load_dataclass_jgzip, FrameAnnotation, SequenceAnnotation
)
from typing import List
import re
import json

In [30]:
CO3DV2_DATASET_ROOT = "/home/gsplat/HD/code/KeyGS/dataset/CFGS_Co3d_inter"
category_name = "teddybear"
sequence_name = "34_1403_4393"

category_frame_annotations = load_dataclass_jgzip(
    f"{CO3DV2_DATASET_ROOT}/{category_name}/frame_annotations.jgz", List[FrameAnnotation]
)
# category_sequence_annotations = load_dataclass_jgzip(
#     f"{CO3DV2_DATASET_ROOT}/{category_name}/{sequence_name}/sequence_annotations.jgz", List[SequenceAnnotation]
# )

In [31]:
selected_frame_annotations = [item for item in category_frame_annotations if item.sequence_name == sequence_name]

In [32]:
selected_frame_annotations[0]

FrameAnnotation(sequence_name='34_1403_4393', frame_number=1, frame_timestamp=0.0, image=ImageAnnotation(path='teddybear/34_1403_4393/images/frame000001.jpg', size=(858, 481)), depth=DepthAnnotation(path='teddybear/34_1403_4393/depths/frame000001.jpg.geometric.png', scale_adjustment=1.0, mask_path='teddybear/34_1403_4393/depth_masks/frame000001.png'), mask=MaskAnnotation(path='teddybear/34_1403_4393/masks/frame000001.png', mass=39806.0), viewpoint=ViewpointAnnotation(R=((0.8687542676925659, 0.396296888589859, 0.29700979590415955), (-0.4568174481391907, 0.4096222221851349, 0.789637565612793), (0.19126909971237183, -0.8216802477836609, 0.5368964076042175)), T=(0.3559315800666809, 0.08288713544607162, 15.170653343200684), focal_length=(3.376338481903076, 3.376338481903076), principal_point=(-0.0, -0.0), intrinsics_format='ndc_isotropic'), meta={'frame_type': 'train_known', 'frame_splits': ['multisequence_teddybear_train_known'], 'eval_batch_maps': []})

In [33]:
formatted_data_list = [{
    "id": item.frame_number,
    "img_name": "frame" + re.search(r'frame(\d+)\.jpg', item.image.path).group(1) + ".jpg",
    "width": item.image.size[1],
    "height": item.image.size[0],
    "w2c_position": list(item.viewpoint.T),
    "w2c_rotation": [list(row) for row in item.viewpoint.R],
    "fy": item.viewpoint.focal_length[1],
    "fx": item.viewpoint.focal_length[0]
} for index, item in enumerate(selected_frame_annotations)]

In [34]:
formatted_data_list

[{'id': 1,
  'img_name': 'frame000001.jpg',
  'width': 481,
  'height': 858,
  'w2c_position': [0.3559315800666809,
   0.08288713544607162,
   15.170653343200684],
  'w2c_rotation': [[0.8687542676925659,
    0.396296888589859,
    0.29700979590415955],
   [-0.4568174481391907, 0.4096222221851349, 0.789637565612793],
   [0.19126909971237183, -0.8216802477836609, 0.5368964076042175]],
  'fy': 3.376338481903076,
  'fx': 3.376338481903076},
 {'id': 2,
  'img_name': 'frame000002.jpg',
  'width': 481,
  'height': 858,
  'w2c_position': [0.36791685223579407,
   0.10725502669811249,
   15.16757583618164],
  'w2c_rotation': [[0.8654226064682007,
    0.40060409903526306,
    0.3009319603443146],
   [-0.46219199895858765, 0.40639713406562805, 0.7881750464439392],
   [0.1934482753276825, -0.8211928606033325, 0.5368613004684448]],
  'fy': 3.3829312324523926,
  'fx': 3.3829312324523926},
 {'id': 3,
  'img_name': 'frame000003.jpg',
  'width': 481,
  'height': 858,
  'w2c_position': [0.382043808698654

In [35]:
# Writing JSON data
with open(f"{CO3DV2_DATASET_ROOT}/{category_name}/{sequence_name}/camera_poses.json", 'w') as f:
    json.dump(formatted_data_list, f, indent=4)